Importación de librerías y Configuración de Rutas Relativas
Arrancamos cargando las librerías necesarias y apuntando directamente al JSON crudo.

In [ ]:

import os

# Configuración de rutas relativas seguras
PATH_RAW = os.path.join("..", "data", "raw", "streaming_users_dirty.json")

# Carga del dataset original exigido
df = pd.read_json(PATH_RAW)
print(f"Dataset cargado para limpieza. Filas iniciales: {df.shape[0]}")

Dataset cargado para limpieza. Filas iniciales: 8160


Eliminación de Registros Duplicados
Acá limpiamos las filas repetidas para que no alteren las métricas y guardamos cuántas eliminamos.

In [14]:
# Guardamos la cantidad antes de borrar
filas_antes = df.shape[0]

# Eliminamos duplicados basados en el ID de usuario
df = df.drop_duplicates(subset=['user_id'])

filas_despues = df.shape[0]
print(f"Se eliminaron {filas_antes - filas_despues} registros duplicados.")

Se eliminaron 160 registros duplicados.


Tratamiento de Valores Nulos (Missing Values)
Acá aplicás la estrategia de limpieza que definiste (por ejemplo, llenar las edades vacías con la mediana, o borrar filas críticas sin datos).

In [15]:
# Ejemplo estándar: Imputamos la mediana en la edad si faltan datos
if 'age' in df.columns:
    df['age'] = df['age'].fillna(df['age'].median())

# Eliminamos filas donde falten datos esenciales que no se puedan inventar
df = df.dropna(subset=['subscription_plan', 'monthly_watch_time_mins'])
print(f"Tratamiento de nulos completado. Filas resultantes: {df.shape[0]}")

Tratamiento de nulos completado. Filas resultantes: 7807


Registro en la Bitácora de Control (logs/)
Esta es la celda obligatoria donde dejamos asentado en tu archivo de texto/CSV de logs que el proceso se ejecutó correctamente.

In [20]:
from datetime import datetime

# Definimos la ruta del log
PATH_LOGS = os.path.join("..", "logs", "pipeline_log.csv")

# Creamos un pequeño registro
nuevo_log = pd.DataFrame([{
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "registros_iniciales": filas_antes,
    "registros_finales": df.shape[0],
    "estado": "Exitoso"
}])

# Guardamos el log (si ya existe, añade la fila abajo; si no, lo crea)
nuevo_log.to_csv(PATH_LOGS, mode='a', index=False, header=not os.path.exists(PATH_LOGS))
print("Bitácora de control (log) actualizada con éxito.")

Bitácora de control (log) actualizada con éxito.


Exportación del Dataset Curado (El puente con Streamlit)
¡Esta es la celda más importante! Guardamos el resultado final como el CSV limpio que van a leer tus páginas de Streamlit.

In [21]:
# Definimos la ruta de salida en la carpeta processed
PATH_PROCESSED = os.path.join("..", "data", "processed", "dataset_procesado.csv")

# Exportamos a CSV plano optimizado
df.to_csv(PATH_PROCESSED, index=False, encoding='utf-8-sig')
print("¡Dataset curado exportado correctamente en data/processed/dataset_procesado.csv!")

¡Dataset curado exportado correctamente en data/processed/dataset_procesado.csv!


Normalizar Países y Géneros


In [22]:
# Paso 5: Normalizar columna Country (Pasar todo a mayúscula inicial y quitar espacios)
if 'country' in df.columns:
    df['country'] = df['country'].astype(str).str.strip().str.title()
    # Si detectas abreviaturas como 'Arg' o 'Br', puedes usar un .replace() aquí

# Paso 6: Normalizar columna Favorite_genre (Homogeneizar la escritura de categorías)
if 'favorite_genre' in df.columns:
    df['favorite_genre'] = df['favorite_genre'].astype(str).str.strip().str.title()
    
print("Países detectados:", df['country'].unique())
print("Géneros detectados:", df['favorite_genre'].unique())

Países detectados: <ArrowStringArray>
[   'Brasil',  'Colombia',   'Uruguay',      'Perú',     'Chile', 'Argentina',
    'México',    'Brazil',       'Mex',       'Arg',       'Col',    'Mexico',
       'Ury',      'Peru',       'Per',       'Bra',       'Chl']
Length: 17, dtype: str
Géneros detectados: <ArrowStringArray>
[      'Crime',    'Thriller',       'Drama',      'Acción',     'Romance',
     'Comedia',  'Documental',           nan,      'Comedy', 'Documentary',
      'Accion',      'Action',         'Doc',      'Crimen',     'Thriler']
Length: 15, dtype: str


Corregir Edades y Tiempos Negativos/Extremos
Corregimos las edades negativas o mayores a 100, y los minutos de reproducción menores a cero, son considerados ruidos o errores de carga (outliers). Los corregimos usando topes lógicos o la mediana.

In [23]:
# Corregir Edades Inválidas (Filtrar un rango lógico entre 12 y 90 años)
if 'age' in df.columns:
    # Reemplazamos cualquier edad loca (fuera de rango) por la mediana del dataset
    mediana_edad = df['age'].median()
    df.loc[(df['age'] <= 0) | (df['age'] > 100), 'age'] = mediana_edad

#  Corregir tiempos de visualización y tickets de soporte negativos
if 'monthly_watch_time_mins' in df.columns:
    df.loc[df['monthly_watch_time_mins'] < 0, 'monthly_watch_time_mins'] = 0

if 'customer_support_tickets' in df.columns:
    df.loc[df['customer_support_tickets'] < 0, 'customer_support_tickets'] = 0

print("Verificación de numéricas completada.")

Verificación de numéricas completada.


Arreglar Formato de Fecha

In [24]:
# Paso 10: Arreglar last_login_date y convertir a datetime para filtros en Streamlit
if 'last_login_date' in df.columns:
    df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')
    # Si alguna fecha quedó rota por error de conversión, rellenamos con la más común
    if df['last_login_date'].isnull().sum() > 0:
        df['last_login_date'] = df['last_login_date'].fillna(df['last_login_date'].mode()[0])

print("Paso 10: Conversión de fechas completada de forma exitosa.")

Paso 10: Conversión de fechas completada de forma exitosa.


Guardar Bitácora y CSV Limpio

In [25]:
from datetime import datetime

# Pasos 12 y 13: Configuración de rutas de salida, logs y guardado final
PATH_LOGS = os.path.join("..", "logs", "pipeline_log.csv")
PATH_PROCESSED = os.path.join("..", "data", "processed", "dataset_procesado.csv")

# Creamos el registro del log con el estado final
nuevo_log = pd.DataFrame([{
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "registros_finales": df.shape[0],
    "estado": "Exitoso_Con_JSON_Completo"
}])
nuevo_log.to_csv(PATH_LOGS, mode='a', index=False, header=not os.path.exists(PATH_LOGS))

# Exportamos el CSV limpio definitivo que lee tu app de Streamlit
df.to_csv(PATH_PROCESSED, index=False, encoding='utf-8-sig')

print("¡Proceso terminado con éxito! Dataset guardado en data/processed/dataset_procesado.csv")

¡Proceso terminado con éxito! Dataset guardado en data/processed/dataset_procesado.csv


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB
